In [ ]:
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bars
from concurrent.futures import ThreadPoolExecutor, as_completed
import osmnx as ox
from shapely.geometry import box
import geopandas as gpd
import h3

# Function to query landuse for a single area
def query_landuse(geom, tags={"landuse": "retail"}):
    try:
        gdf = ox.features_from_polygon(geom, tags)
        return gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]
    except Exception as e:
        return None

# Split the area into smaller pieces
def split_area(geometry, num_splits=4):
    minx, miny, maxx, maxy = geometry.bounds
    dx = (maxx - minx) / num_splits
    dy = (maxy - miny) / num_splits
    pieces = []
    for i in range(num_splits):
        for j in range(num_splits):
            pieces.append(box(minx + i * dx, miny + j * dy, minx + (i + 1) * dx, miny + (j + 1) * dy))
    return pieces

# Parallel query with progress tracking
def query_with_progress_concurrent(boundary_geometry, num_splits=4, max_workers=4):
    # Split the area into pieces
    pieces = split_area(boundary_geometry, num_splits=num_splits)

    # Use ThreadPoolExecutor with progress tracking
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(query_landuse, piece) for piece in pieces]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Querying Sub-areas"):
            try:
                result = future.result()  # Get the result of the completed task
                results.append(result)    # Append the result to the results list
            except Exception as e:
                print(f"Error: {e}")
            
    combined_gdf = gpd.GeoDataFrame(pd.concat(results, ignore_index=True))
    return combined_gdf


In [ ]:


# Get the UK boundary
uk_boundary = ox.geocode_to_gdf("United Kingdom").geometry[0]
# Query retail landuse data in parallel with progress tracking
retail_landuse = query_with_progress_concurrent(uk_boundary, num_splits=10, max_workers=16)

# Clip the data to the UK boundary and remove duplicates
retail_landuse = (
    gpd.clip(retail_landuse, uk_boundary)
    [['geometry', 'landuse', 'name']]
    .drop_duplicates()
)
#save to file (geoparquet)
retail_landuse.to_parquet("data/input_files/retail_landuse.parquet")

Querying Sub-areas: 100%|██████████| 100/100 [00:22<00:00,  4.50it/s]


In [ ]:


# Function to convert polygon or multipolygon to H3 indices
def get_h3_indices(geometry, resolution=13):
    # Check if the geometry is a Polygon or MultiPolygon
    if geometry.geom_type == 'Polygon':
        coords = [(lat, lng) for lng, lat in geometry.exterior.coords]
    elif geometry.geom_type == 'MultiPolygon':
        coords = []
        # Use geoms to iterate over individual polygons in the MultiPolygon
        for poly in geometry.geoms:
            coords.extend([(lat, lng) for lng, lat in poly.exterior.coords])
    else:
        raise ValueError(f"Geometry type {geometry.geom_type} not supported")

    # Create a LatLngPoly object
    poly = h3.LatLngPoly(coords)
    # Convert the polygon to H3 cells at the specified resolution
    return h3.h3shape_to_cells(poly, res=resolution)

# # Apply the function to the GeoDataFrame to get H3 indices at resolution 13
retail_landuse['h3_indices_res13'] = retail_landuse['geometry'].apply(get_h3_indices, resolution=13)
retail_landuse['h3_indices_res11'] = retail_landuse['geometry'].apply(get_h3_indices, resolution=11)

# Now apply the function to convert the H3 indices to resolution 11
#drop rows with empty h3_indices_res13, these are a handful of very small polygons
retail_landuse = retail_landuse[retail_landuse['h3_indices_res13'].apply(len) > 0]
retail_landuse['h3_indices_res11_parents'] = retail_landuse['h3_indices_res13'].apply(lambda h3_cells: [h3.cell_to_parent(cell, 11) for cell in h3_cells])
#explode h3_indices_res11_parents and rename to h3_address
retail_landuse_h3 = retail_landuse.explode('h3_indices_res11_parents').rename(columns={'h3_indices_res11_parents': 'h3_11'}).drop(columns=['h3_indices_res13', 'h3_indices_res11',"name","geometry"])
#drop repeats

retail_landuse_h3 = retail_landuse_h3.drop_duplicates()

#save to file (parquet)
retail_landuse_h3.to_parquet("data/input_files/retail_landuse_h3.parquet")